In [121]:
import psycopg2
import os
import pandas as pd
import ast
import re

from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()
pd.set_option('display.max_rows', 100000)
pd.set_option("max_colwidth", 500)
KEY_POSTGRES = os.getenv("KEY_POSTGRES")

In [64]:
DF_input = input().strip()

 C:\Users\svalb\OneDrive\Escritorio\Data_40_years_cancer_studies\preparedInstitutionsTestDatabaseAccess.csv


In [65]:
df = pd.read_csv(DF_input)

In [66]:
s = df.iloc[2]["NER_BERT"]

In [67]:
df.iloc[2]["Institutions_man"]

"['Instituto de Investigação e Inovação em Saúde', 'University of Porto', 'University of British Columbia', 'BC Cancer', 'Institute of Molecular Pathology and Immunology']"

In [68]:
cleaned = re.sub(r"np\.float\d+\((.*?)\)", r"\1", s)
cleaned

"[{'entity_group': 'INS', 'score': 0.99988544, 'word': 'University of British Columbia', 'start': 32, 'end': 62}, {'entity_group': 'SUBC', 'score': 0.99957687, 'word': 'Vancouver', 'start': 64, 'end': 73}, {'entity_group': 'COU', 'score': 0.9991843, 'word': 'Canada', 'start': 75, 'end': 81}, {'entity_group': 'INS', 'score': 0.9996547, 'word': 'Hereditary Cancer', 'start': 112, 'end': 129}, {'entity_group': 'INS', 'score': 0.9165015, 'word': 'BC Cancer', 'start': 131, 'end': 140}, {'entity_group': 'INS', 'score': 0.9998889, 'word': 'University of British Columbia', 'start': 196, 'end': 226}, {'entity_group': 'SUBC', 'score': 0.99966955, 'word': 'Vancouver', 'start': 228, 'end': 237}, {'entity_group': 'COU', 'score': 0.9899244, 'word': 'Canada', 'start': 239, 'end': 245}, {'entity_group': 'INS', 'score': 0.9998265, 'word': 'Instituto de Investigação e Inovação em Saúde', 'start': 253, 'end': 298}, {'entity_group': 'INS', 'score': 0.9998478, 'word': 'Institute of Molecular Pathology and I

In [91]:
query = ast.literal_eval(cleaned)[6]["word"]
query

'Vancouver'

In [92]:
connection = psycopg2.connect(database="ror_institutions", user="postgres", password=KEY_POSTGRES, host="localhost", port=5432)
cursor = connection.cursor()
cursor.execute("SELECT DISTINCT ON (institution_id) name FROM public.institution_names WHERE name = %s;", (query,))
record = cursor.fetchall()
record

[]

In [53]:
ast.literal_eval(cleaned)[0]

{'entity_group': 'INS',
 'score': 0.99988544,
 'word': 'University of British Columbia',
 'start': 32,
 'end': 62}

In [106]:
df['Institutions_db'] = [[] for _ in range(len(df))]
df['Countries_db'] = [[] for _ in range(len(df))]

for row in tqdm(range(len(df))):
    cleaned = re.sub(r"np\.float\d+\((.*?)\)", r"\1", df.iloc[row]["NER_BERT"])
    for NER_result in ast.literal_eval(cleaned):
        if NER_result['entity_group'] == 'INS':
            query = NER_result['word']
            connection = psycopg2.connect(database="ror_institutions", user="postgres", password=KEY_POSTGRES, host="localhost", port=5432)
            cursor = connection.cursor()
            cursor.execute(f"SELECT * FROM institution_names WHERE name = '{query}';")
            record = cursor.fetchall()
            institutions_db = []
            for el in record:
                institutions_db.append(el[3])
            institutions_db = list(set(institutions_db))
            if len(institutions_db) == 1:
                df.iloc[row]['Institutions_db'].append(institutions_db)
                i = 0
                while i == 0:
                    for el in record:
                        if institutions_db[0] in el:
                            new_query = el[2]
                            connection = psycopg2.connect(database="ror_institutions", user="postgres", password=KEY_POSTGRES, host="localhost", port=5432)
                            cursor = connection.cursor()
                            cursor.execute(f"SELECT country_name FROM institution_locations WHERE country_name = '{new_query}';")
                            new_record = cursor.fetchall()
                            if len(new_record) > 0:
                                df.iloc[row]['Countries_db'].append(new_record[0][5])
                                i += 1

  2%|█▌                                                                              | 2/103 [02:20<1:58:08, 70.18s/it]


KeyboardInterrupt: 

# WITH CACHE

In [ ]:
# Create empty columns
df['Institutions_db'] = [[] for _ in range(len(df))]
df['Countries_db'] = [[] for _ in range(len(df))]

# Open ONE connection
connection = psycopg2.connect(
    database="ror_institutions",
    user="postgres",
    password=KEY_POSTGRES,
    host="localhost",
    port=5432
)
cursor = connection.cursor()

# Cache to avoid repeated DB hits
institution_cache = {}

for row in tqdm(range(len(df))):
    print(row)
    if pd.isna(df.at[row, "NER_BERT"]):    
        continue
        
    cleaned = re.sub(r"np\.float\d+\((.*?)\)", r"\1", df.at[row, "NER_BERT"])
    ner_results = ast.literal_eval(cleaned)
    
    for ner in ner_results:
        if ner.get('entity_group') != 'INS':
            continue
    
        query = ner.get('word')

        
        if '\'' in query:
            query = re.sub("\'", "\'\'", query)
    
        # ---- Cache lookup ----
        if query in institution_cache:
            institutions_db, countries_db = institution_cache[query]
        else:
            # Parameterized query
            cursor.execute(
                "SELECT * FROM institution_names WHERE name = %s;",
                (query,)
            )
            records = cursor.fetchall()
    
            institutions_db = list({el[1] for el in records})
            
            institutions_id = list({el[0] for el in records})
            print(institutions_id)
            
            countries_db = []
    
            if len(set(institutions_id)) == 1:
    
                # Find related country via institution_locations
                for el in records:
                    #if institution_id in el:
                    new_query = el[0]
    
                    cursor.execute(
                        "SELECT country_name FROM institution_locations WHERE institution_id = %s;",
                        (new_query,)
                    )
                    location_records = cursor.fetchall()
    
                    if location_records:
                        countries_db.append(location_records[0][0])
                        break

            
            institution_cache[query] = (institutions_db, countries_db)
    
        # Append results safely
        if institutions_db:
            df.at[row, 'Institutions_db'].append(institutions_db[0])
        if countries_db:
            df.at[row, 'Countries_db'].append(countries_db[0])
            
cursor.close()
connection.close()

# NO CACHE

In [ ]:
# Create empty columns
df['Institutions_db_id'] = [[] for _ in range(len(df))]
df['Institutions_db_name'] = [[] for _ in range(len(df))]
df['Countries_db_country'] = [[] for _ in range(len(df))]

# Open ONE connection
connection = psycopg2.connect(
    database="ror_institutions",
    user="postgres",
    password=KEY_POSTGRES,
    host="localhost",
    port=5432
)
cursor = connection.cursor()

for row in tqdm(range(len(df))):
    found_institutions = []

    if pd.isna(df.at[row, "NER_BERT"]):    
        continue

    cleaned = re.sub(r"np\.float\d+\((.*?)\)", r"\1", df.at[row, "NER_BERT"])
    ner_results = ast.literal_eval(cleaned)

    subc_ner_results = [ner.get('word') for ner in ner_results if ner.get('entity_group') == 'SUBC']
    cou_ner_results = [ner.get('word') for ner in ner_results if ner.get('entity_group') == 'COU']

    for ner in ner_results:
        if ner.get('entity_group') != 'INS':
            continue
    
        query = ner.get('word')

        if '\'' in query:
            query = re.sub("\'", "\'\'", query)
        
        # Parameterized query
        cursor.execute(
            "SELECT institution_id, name FROM institution_names WHERE name = %s;",
            (query,)
        )
        records = cursor.fetchall()

        candidates = {
            el[0]: el[1]
            for el in records
        }

        if len(candidates) == 1:
            new_query = list(candidates.keys())[0]
    
            cursor.execute(
                "SELECT country_name FROM institution_locations WHERE institution_id = %s;",
                (new_query,)
            )
            
            location_records = cursor.fetchall()
    
            if location_records:
                found_institutions.append((new_query, list(candidates.values())[0], location_records[0][0]))
                continue

        
        elif len(candidates) == 0:
            # Call the database again with a fuzzy string match
            cursor.execute(
                "SELECT institution_id, name FROM institution_names WHERE levenshtein(name, %s) <= 2;",
                (query,)
            )
            records = cursor.fetchall()
            
            candidates = {
                el[0]: el[1]
                for el in records
            }
            
            if len(candidates) == 1:
                
                # Find related country via institution_locations
                new_query = list(candidates.keys())[0]
    
                cursor.execute(
                    "SELECT city_name, country_name, subdivision_name FROM institution_locations WHERE institution_id = %s;",
                    (new_query,)
                )
                location_records = cursor.fetchall()
    
                if location_records and (location_records[0][1] in cou_ner_results):
                    found_institutions.append((new_query, list(candidates.values())[0], location_records[0][0]))
                    continue

                elif location_records and ((location_records[0][0] in subc_ner_results) or (location_records[0][2] in subc_ner_results)):
                    found_institutions.append((new_query, list(candidates.values())[0], location_records[0][0]))
                    continue
                else:
                    continue

            
            elif len(candidates) == 0:
                continue

            elif len(candidates) > 1:
                matching_candidates = []

                for inst_id, inst_name in candidates.items():
                    new_query = inst_id    

                    cursor.execute(
                        "SELECT city_name, country_name, subdivision_name FROM institution_locations WHERE institution_id = %s;",
                        (new_query,)
                    )

                    location_records = cursor.fetchall()

                    if location_records and (location_records[0][1] in cou_ner_results):
                        matching_candidates.append((inst_id, 
                                                    inst_name,
                                                    location_records[0][0]))
                    
                    elif location_records and ((location_records[0][0] in subc_ner_results) or (location_records[0][2] in subc_ner_results)):
                        matching_candidates.append((inst_id, 
                                                    inst_name,
                                                    location_records[0][0]))
                    

                if len(matching_candidates) == 1:
                    found_institutions.append(matching_candidates[0])

                # Grow here to add dissambiguation via hierarchies of institutions



        elif len(candidates) > 1:
            matching_candidates = []

            for inst_id, inst_name in candidates.items():
                new_query = inst_id    


                cursor.execute(
                    "SELECT city_name, country_name, subdivision_name FROM institution_locations WHERE institution_id = %s;",
                    (new_query,)
                )

                location_records = cursor.fetchall()

                if location_records and (location_records[0][1] in cou_ner_results):
                    matching_candidates.append((new_query,
                                                inst_name,
                                                location_records[0][0]))
                    
                elif location_records and ((location_records[0][0] in subc_ner_results) or (location_records[0][2] in subc_ner_results)):
                    matching_candidates.append((new_query, 
                                                inst_name,
                                                location_records[0][0]))
                    

            if len(matching_candidates) == 1:
                found_institutions.append(matching_candidates[0])

            # Grow here to add dissambiguation via hierarchies of institutions


    
    if found_institutions:
        ids, names, countries = zip(*found_institutions)

        df.at[row, 'Institutions_db_id'] = list(ids)
        df.at[row, 'Institutions_db_name'] = list(names)
        df.at[row, 'Countries_db_country'] = list(countries)


cursor.close()
connection.close()